# Thrust chamber design with `h2cea`

This notebook takes a thrust chamber from the requirement to a checked first design, using the
`h2cea` library of `e2-propulsion-tools` for every physical calculation. The notebook itself only
holds the design inputs, the order of the steps and the figures.

| Step | Content | `h2cea` modules |
|---|---|---|
| 1 | Setup and run record | `config` |
| 2 | Design inputs (the only cell to edit) | `config` (baseline defaults) |
| 3 | Thermochemistry: propellant cards, mixture ratio sweep | `propellants`, `cea_si` |
| 4 | Design point: mixture ratio, throat, mass flows | `operating_line`, `cea_si` |
| 5 | Throttle line | `operating_line` |
| 6 | Expansion ratio and flow separation | `cea_si`, `gasdyn` |
| 7 | Chamber and nozzle contour | `contour`, `gasdyn`, `cea_si` |
| 8 | Gas-side heat load (Bartz, fixed wall temperature) | `bartz` |
| 9 | Regenerative cooling: jacket geometry and coupled march | `jacket`, `regen`, `coolant`, `n2o` |
| 10 | Wall stresses | `structure`, `materials` |
| 11 | Design summary and export | – |

**How to use it.** Put the notebook anywhere inside `e2-propulsion-tools/` (or set the environment
variable `H2CEA_DIR` to the `h2cea/` folder), edit the `Spec` in step 2 and run all cells. The
defaults of the `Spec` are the shared baseline (`baseline/h2_baseline.yaml`, read through
`h2cea.config`), so an unedited run reproduces the E2 flight engine. For a different engine,
override the fields in step 2; nothing in the repository is changed. Results are written to
`results/thrust_chamber_design/<name>/` next to the notebook.

**Conventions.** SI units inside the library, with pressure in bar at the function interfaces.
`x` runs along the axis from the injector face. Efficiencies are split as in the baseline:
$\eta_{c^*}$ on the characteristic velocity and $\eta_{C_F}$ on the ideal *vacuum* thrust
coefficient, so the delivered thrust is

$$F = \eta_{C_F}\,C_{F,\mathrm{vac}}\,p_c\,A_t - p_a\,A_e , \qquad
\dot m = \frac{p_c\,A_t}{\eta_{c^*}\,c^*} .$$

## 1 Setup and run record

The cell locates the `h2cea` package, imports it and records the versions and the git commit, so
that numbers quoted from this notebook can be traced.

In [ ]:
import os, sys, json, hashlib, subprocess, datetime as dt, warnings
from dataclasses import dataclass, asdict, replace
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from scipy.optimize import brentq


def find_h2cea():
    # folder that contains the h2cea package: $H2CEA_DIR, or searched upwards from here
    env = os.environ.get("H2CEA_DIR")
    here = Path.cwd().resolve()
    candidates = [Path(env)] if env else []
    for p in [here, *here.parents]:
        candidates += [p, p / "h2cea", p / "e2-propulsion-tools" / "h2cea"]
    for c in candidates:
        if (c / "h2cea" / "cea_si.py").exists():
            return c
    raise FileNotFoundError("h2cea not found. Put this notebook inside e2-propulsion-tools/ "
                            "or set H2CEA_DIR to the e2-propulsion-tools/h2cea folder.")


H2CEA_DIR = find_h2cea()
sys.path.insert(0, str(H2CEA_DIR))

from h2cea import plotstyle as PS            # shared figure style (selects the Agg backend)
%matplotlib inline
import matplotlib.pyplot as plt

import rocketcea, CoolProp
from h2cea import config as C
from h2cea import gasdyn as GD, materials as MAT, regen, structure as ST, n2o as N2O
from h2cea.cea_si import point, pinj_over_pcomb, species
from h2cea.propellants import n2o_card, ethanol_card
from h2cea.operating_line import size_throat, state
from h2cea.contour import e2_contour, wall_stations, L_star, chamber_volume, cone_length
from h2cea.bartz import bartz_profile
from h2cea.jacket import Jacket, channels
from h2cea.coolant import h_from_Tp

pd.set_option("display.width", 160, "display.max_columns", 30, "display.max_colwidth", None)


def git_commit(path):
    try:
        return subprocess.run(["git", "-C", str(path), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True, timeout=10).stdout.strip() or "unknown"
    except Exception:
        return "unknown"


RUN = dict(date=dt.datetime.now().isoformat(timespec="seconds"), h2cea_dir=str(H2CEA_DIR),
           h2cea_commit=git_commit(H2CEA_DIR), rocketcea=rocketcea.__version__,
           coolprop=CoolProp.__version__, numpy=np.__version__,
           baseline=f"{C.BASELINE['meta']['engine']}, revision {C.BASELINE['meta']['revision']}")
pd.Series(RUN, name="run record").to_frame()

## 2 Design inputs

All inputs are in the `Spec` below. The defaults are the baseline values from `h2cea.config`; change
a field in the `spec = Spec(...)` call at the end of the cell to study another design.

Two ways to fix the mixture ratio are supported:

- **Fixed fuel flow** (`mdot_fuel` set, the E2 architecture): the fuel cavitating venturi fixes the
  fuel flow and the oxidiser is throttled, so the mixture ratio at rated thrust follows from the
  sizing and moves along the throttle line.
- **Fixed mixture ratio** (`mdot_fuel=None`): `mr` is used at every thrust level.

The Rao angles `th_n` and `th_e` are chart values for the chosen area ratio and bell fraction.
The defaults belong to $\varepsilon = 4$ with an 80 % bell; read them again from the chart when
the area ratio changes.

In [ ]:
@dataclass(frozen=True)
class Spec:
    name: str = "E2-flight"
    # ---- requirement --------------------------------------------------------------
    F: float = C.F_MAX                    # N, sea-level thrust at 100 %
    pc: float = C.PC_MAX                  # bar, chamber pressure at 100 %
    throttle_min: float = C.THROTTLE_FLOOR   # fraction of F
    mdot_fuel: float | None = C.MDOT_FUEL    # kg/s fixed fuel flow; None = fixed mixture ratio
    mr: float = 4.0                       # O/F, used only when mdot_fuel is None
    eta_cstar: float = C.ETA_CSTAR        # c* efficiency
    eta_cf_vac: float = C.ETA_CF_VAC      # on the ideal vacuum thrust coefficient
    # ---- propellants ----------------------------------------------------------------
    T_ox: float = C.T_OX_NOM              # K, N2O in the tank
    p_tank: float = C.P_TANK              # bar, N2O tank pressure
    ethanol_wt: float = C.ETHANOL_WT      # wt% ethanol in the fuel
    # ---- nozzle -----------------------------------------------------------------------
    eps: float = C.EPS                    # area ratio Ae/At
    bell: float = C.BELL_FRACTION         # Rao bell length, fraction of the 15 deg cone
    th_n: float = C.THETA_N               # deg, Rao initial angle (chart value for eps, bell)
    th_e: float = C.THETA_E               # deg, Rao exit angle (chart value for eps, bell)
    # ---- chamber ----------------------------------------------------------------------
    CR: float = 12.0                      # contraction ratio Ac/At
    L_star: float = 1.15                  # m, characteristic length
    th_conv: float = C.THETA_CONV         # deg, convergent half angle
    R1_over_Rc: float = 1.7               # blend radius cylinder -> cone, in chamber radii
    Ru_over_Rt: float = 1.5               # throat upstream radius, in throat radii
    Rd_over_Rt: float = 0.382             # throat downstream radius, in throat radii
    # ---- cooling jacket -----------------------------------------------------------------
    n_ch: int = C.N_CH                    # channels
    t_wall: float = C.T_WALL              # m, hot wall
    t_rib: float = C.T_RIB                # m, rib (constant)
    h_ch: float = C.H_CH                  # m, channel height (constant)
    t_closeout: float = C.T_CLOSEOUT      # m
    roughness: float = C.ROUGHNESS        # m, absolute channel roughness
    material: str = C.WALL_MATERIAL
    # ---- cooling analysis ---------------------------------------------------------------
    p_inj_in: float = C.P_INJ_IN_NOM      # bar, oxidiser injector inlet (= jacket outlet) at 100 %
    tp_model: str = C.N2O_TP_MODEL        # coolant-side model for boiling N2O (twophase.MODELS)
    bartz_factor: float = C.BARTZ_FACTOR  # gas-side multiplier (calibrate with calorimetry)
    Twg_bartz: float = 1000.0             # K, wall temperature for the uncoupled Bartz load (step 8)
    n_stations: int = 300                 # wall stations of the march
    return_jacket_heat: bool = True       # N2O carries the jacket heat into the chamber
    # ---- water-cooled test configuration (optional, step 9.4) ---------------------------
    water_mdot: float = C.WATER_MDOT      # kg/s
    water_p_in: float = C.WATER_P_IN      # bar
    water_T_in: float = C.WATER_T_IN      # K


spec = Spec()        # e.g. Spec(name="study-eps35", eps=3.5, th_n=20.0, th_e=14.5)

SPEC_SHA = hashlib.sha256(json.dumps(asdict(spec), sort_keys=True).encode()).hexdigest()[:8]
OUT = Path.cwd() / "results" / "thrust_chamber_design" / spec.name
OUT.mkdir(parents=True, exist_ok=True)
P_AMB = C.P_AMB      # bar, from the baseline; the library uses it for all sea-level values
FIXED_FUEL = spec.mdot_fuel is not None

print(f"Spec '{spec.name}'  sha {SPEC_SHA}  ->  {OUT}")
pd.Series(asdict(spec), name="value").to_frame()

## 3 Thermochemistry

### 3.1 Propellant cards

CEA solves an adiabatic equilibrium at constant pressure, so the result depends on the enthalpy the
reactants bring in. RocketCEA's built-in `N2O` card is ideal gas at 298 K. The oxidiser of this
engine leaves the tank as a compressed liquid, which is lower in enthalpy. `propellants.n2o_card()`
registers a card at the real tank state (CoolProp reference equation of state), and this notebook
uses that card everywhere. The comparison below shows what the built-in gas card would overstate.

In [ ]:
OX = n2o_card(spec.T_ox, spec.p_tank)          # liquid N2O at the tank state
FUEL = ethanol_card(spec.ethanol_wt)
dh_card = N2O.h_TP(298.15, 0.001) - N2O.h_TP(spec.T_ox, spec.p_tank)   # J/kg below ideal gas


def cea(pc, mr, eps=None, ox=None, fuel=None):
    # h2cea.cea_si.point with the propellants of this notebook; inputs rounded for the cache
    return point(round(float(pc), 4), round(float(mr), 5), round(float(eps or spec.eps), 4),
                 P_AMB, ox or OX, fuel or FUEL)


mr_ref = spec.mr if not FIXED_FUEL else 4.0
liq, gas = cea(spec.pc, mr_ref), cea(spec.pc, mr_ref, ox="N2O")
cards = pd.DataFrame({
    "liquid card (tank state)": [liq.Tc, liq.cstar, liq.isp_vac, liq.isp_amb],
    "built-in gas card": [gas.Tc, gas.cstar, gas.isp_vac, gas.isp_amb]},
    index=["Tc  K", "c*  m/s", "Isp vac  s", "Isp sea level, ideal  s"])
cards["gas card overstates by  %"] = 100 * (cards.iloc[:, 1] / cards.iloc[:, 0] - 1)
print(f"Cards: oxidiser '{OX}', fuel '{FUEL}'.  Tank-state N2O is {dh_card/1e3:.0f} kJ/kg below "
      f"the ideal gas at 298 K.  Comparison at pc {spec.pc:g} bar, O/F {mr_ref:g}, eps {spec.eps:g}:")
cards.round(2)

### 3.2 Mixture ratio sweep

Ideal (CEA, shifting equilibrium) values against the mixture ratio at the rated chamber pressure and
at half of it. The specific impulse peaks on the fuel-rich side of the temperature peak. The
difference between the equilibrium and the frozen specific impulse is the most the recombination in
the nozzle can contribute.

In [ ]:
MR_SWEEP = np.round(np.arange(2.0, 7.01, 0.1), 2)
PC_SWEEP = [spec.pc, round(0.5 * spec.pc, 3)]
sweep = pd.DataFrame([dict(pc=pc, MR=m, **{k: getattr(cea(pc, m), k) for k in
                      ("Tc", "cstar", "isp_vac", "isp_vac_frozen", "isp_amb", "gam_fr_c", "mw_c", "pe")})
                      for pc in PC_SWEEP for m in MR_SWEEP])

fig, ax = plt.subplots(1, 3, figsize=(13.5, 3.8))
for pc, col in zip(PC_SWEEP, (PS.BLUE, PS.ORANGE)):
    d = sweep[sweep.pc == pc]
    ax[0].plot(d.MR, d.Tc, color=col, label=f"{pc:g} bar")
    ax[1].plot(d.MR, d.cstar, color=col, label=f"{pc:g} bar")
    ax[2].plot(d.MR, d.isp_vac, color=col, label=f"vacuum, equilibrium, {pc:g} bar")
    ax[2].plot(d.MR, d.isp_vac_frozen, color=col, ls=":", label=f"vacuum, frozen at throat, {pc:g} bar")
    ax[2].plot(d.MR, d.isp_amb, color=col, ls="--", label=f"sea level, attached, {pc:g} bar")
ax[0].set(title="Chamber temperature", xlabel="O/F", ylabel="K")
ax[1].set(title="Characteristic velocity (ideal)", xlabel="O/F", ylabel="m/s")
ax[2].set(title=f"Ideal specific impulse, eps {spec.eps:g}", xlabel="O/F", ylabel="s")
ax[0].legend(); ax[2].legend(fontsize=7.5)
fig.tight_layout()

d = sweep[sweep.pc == spec.pc]
best = {k: d.loc[d[k].idxmax()] for k in ("Tc", "cstar", "isp_vac", "isp_amb")}
pd.DataFrame({"maximum": {k: v[k] for k, v in best.items()},
              "at O/F": {k: v.MR for k, v in best.items()}}).rename(
    index={"Tc": "Tc  K", "cstar": "c*  m/s", "isp_vac": "Isp vac  s", "isp_amb": "Isp sea level  s"}).round(2)

## 4 Design point: mixture ratio, throat and mass flows

With the fuel flow fixed, `operating_line.size_throat()` finds the mixture ratio and the throat area
that give the rated thrust at the rated chamber pressure. With a fixed mixture ratio the throat
follows directly from the thrust equation above. `operating_line.state()` then returns the
delivered state of the sized engine; it is used as the single engine model from here on.

In [ ]:
KW = dict(eta_c=spec.eta_cstar, eta_f=spec.eta_cf_vac, fuel=FUEL)


def size(eps=None, ox=None):
    # mixture ratio and throat area (m2) for the rated thrust at the rated chamber pressure
    eps = eps or spec.eps
    if FIXED_FUEL:
        return size_throat(F=spec.F, pc=spec.pc, eps=eps, mf=spec.mdot_fuel, ox=ox or OX, **KW)
    p = cea(spec.pc, spec.mr, eps, ox=ox)
    At = spec.F / (spec.eta_cf_vac * p.cf_vac * spec.pc * 1e5 - P_AMB * 1e5 * eps)
    return spec.mr, At


def op(mox, At, eps=None, ox=None):
    # delivered engine state at an oxidiser flow (h2cea.operating_line.state)
    mf = spec.mdot_fuel if FIXED_FUEL else mox / spec.mr
    return state(mox, At, eps=eps or spec.eps, mf=mf, ox=ox or OX, **KW)


def mox_for_thrust(F, At, mox_ref, eps=None, ox=None):
    return brentq(lambda m: op(m, At, eps, ox)["F"] - F, 0.15 * mox_ref, 1.5 * mox_ref, xtol=1e-6)


MR_D, AT = size()
RT = float(np.sqrt(AT / np.pi))
p_d = cea(spec.pc, MR_D)
mdot_d = spec.pc * 1e5 * AT / (spec.eta_cstar * p_d.cstar)
MOX_D = mdot_d * MR_D / (1 + MR_D)
DP = op(MOX_D, AT)                  # design point

design_point = pd.Series({
    "thrust, sea level  N": DP["F"], "chamber pressure  bar": DP["pc"], "mixture ratio O/F": DP["MR"],
    "total flow  kg/s": DP["mdot"], "oxidiser flow  kg/s": DP["mox"], "fuel flow  kg/s": DP["mf"],
    "throat diameter  mm": 2e3 * RT, "throat area  mm2": AT * 1e6,
    "exit diameter  mm": 2e3 * RT * np.sqrt(spec.eps),
    "Isp delivered, sea level  s": DP["isp"], "Isp ideal, sea level  s": DP["isp_ideal"],
    "Isp efficiency": DP["eta_isp"], "c* ideal  m/s": DP["cstar_ideal"],
    "c* delivered  m/s": spec.eta_cstar * DP["cstar_ideal"], "CF vac ideal": DP["cf_vac"],
    "Tc  K": DP["Tc"], "exit pressure  bar": DP["pe"], "exit Mach": DP["Me"],
    "gamma frozen, chamber": DP["gam_fr"], "molar mass, chamber  kg/kmol": DP["mw"]}, name="design point")
assert abs(DP["F"] - spec.F) < 1e-3 * spec.F and abs(DP["pc"] - spec.pc) < 1e-3 * spec.pc
design_point.to_frame().round(4)

In [ ]:
# Combustion products at the design point (mole fractions above 0.5 %)
# RocketCEA returns the stations as [injector, chamber, throat, exit]: where = 1, 2, 3
sp = pd.DataFrame({w: species(spec.pc, round(MR_D, 5), spec.eps, OX, FUEL, where=i, min_fraction=5e-3)
                   for i, w in ((1, "chamber"), (2, "throat"), (3, "exit"))}).fillna(0.0)
sp.sort_values("chamber", ascending=False).round(4)

## 5 Throttle line

The delivered state at each thrust level with the throat fixed. With a fixed fuel flow the mixture
ratio falls with thrust, and with it the chamber temperature and the specific impulse. The last
column is the Summerfield check used in the baseline engine table (exit pressure below 0.4 of
ambient).

In [ ]:
THROTTLE = np.round(np.linspace(spec.throttle_min, 1.0, 11), 3)
rows = []
for f in THROTTLE:
    s = op(mox_for_thrust(f * spec.F, AT, MOX_D), AT)
    rows.append(dict(throttle=f, F=s["F"], MR=s["MR"], pc=s["pc"], mox=s["mox"], mf=s["mf"],
                     mdot=s["mdot"], isp=s["isp"], Tc=s["Tc"], pe=s["pe"],
                     separation_risk=s["pe"] < 0.4 * P_AMB))
line = pd.DataFrame(rows).set_index("throttle")
LOW = op(line.mox.iloc[0], AT)       # state at the throttle floor

fig, ax = plt.subplots(1, 4, figsize=(14, 3.4))
for a, (k, t, u) in zip(ax, (("pc", "Chamber pressure", "bar"), ("MR", "Mixture ratio", "O/F"),
                             ("isp", "Delivered Isp, sea level", "s"), ("Tc", "Chamber temperature", "K"))):
    a.plot(100 * line.index, line[k], color=PS.BLUE, marker="o", ms=3)
    a.set(title=t, xlabel="thrust, % of rated", ylabel=u)
fig.tight_layout()
line.round(4)

## 6 Expansion ratio and flow separation

For a sea-level engine that throttles, the area ratio is a trade between the specific impulse at
full thrust, the specific impulse at the throttle floor and the separation margin there. The sweep
re-sizes the throat for every area ratio (same thrust and chamber pressure) and evaluates the
throttle floor on that engine.

Separation criteria from `gasdyn`: Summerfield ($p_\mathrm{sep}/p_a \approx 0.37$) and Schmucker
($p_\mathrm{sep}/p_a = (1.88\,M_e - 1)^{-0.64}$). The nozzle flows full while $p_e/p_a$ stays above
the criterion.

In [ ]:
EPS_SWEEP = np.round(np.arange(2.0, 6.01, 0.25), 2)
rows = []
for e in EPS_SWEEP:
    mr_e, At_e = size(e)
    mox_e = spec.pc * 1e5 * At_e / (spec.eta_cstar * cea(spec.pc, mr_e, e).cstar) * mr_e / (1 + mr_e)
    hi = op(mox_e, At_e, e)
    lo = op(mox_for_thrust(spec.throttle_min * spec.F, At_e, mox_e, e), At_e, e)
    rows.append(dict(eps=e, isp_100=hi["isp"], isp_min=lo["isp"], pe_100=hi["pe"], pe_min=lo["pe"],
                     pc_min=lo["pc"], Me=lo["Me"], schmucker=GD.schmucker_psep_over_pa(lo["Me"]),
                     Dt_mm=2e3 * np.sqrt(At_e / np.pi)))
trade = pd.DataFrame(rows).set_index("eps")
trade["pe_min/pa"] = trade.pe_min / P_AMB
trade["full flow at floor"] = trade["pe_min/pa"] > np.maximum(trade.schmucker, GD.summerfield_psep_over_pa())

tmin = f"{100 * spec.throttle_min:.0f} %"
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(trade.index, trade.isp_100, color=PS.BLUE, label="100 %")
ax[0].plot(trade.index, trade.isp_min, color=PS.ORANGE, label=tmin)
ax[0].axvline(spec.eps, color=PS.MUTED, lw=1, ls="--")
ax[0].set(title="Delivered Isp at sea level", xlabel="area ratio", ylabel="s"); ax[0].legend()
ax[1].plot(trade.index, trade.pe_100 / P_AMB, color=PS.BLUE, label="pe/pa at 100 %")
ax[1].plot(trade.index, trade["pe_min/pa"], color=PS.ORANGE, label=f"pe/pa at {tmin}")
ax[1].plot(trade.index, trade.schmucker, color=PS.RED, ls=":", label=f"Schmucker, {tmin}")
ax[1].axhline(GD.summerfield_psep_over_pa(), color=PS.RED, ls="--", lw=1.2, label="Summerfield")
ax[1].axvline(spec.eps, color=PS.MUTED, lw=1, ls="--")
ax[1].set(title="Exit pressure and separation criteria", xlabel="area ratio", ylabel="p / pa"); ax[1].legend()
fig.tight_layout()
trade.round(3)

## 7 Chamber and nozzle contour

`contour.e2_contour()` builds the wall from a cylinder, a blend arc, a conical convergent, the two
throat arcs and a Rao parabola (quadratic Bézier between the inflection point and the exit). The
chamber radius follows from the contraction ratio, and the cylinder length is solved so that the
volume from the injector face to the throat gives the target characteristic length
$L^* = V_c / A_t$.

In [ ]:
RC = RT * np.sqrt(spec.CR)
GEO = dict(eps=spec.eps, Rt=RT, Rc=RC, R1=spec.R1_over_Rc * RC, th_conv=spec.th_conv,
           Ru=spec.Ru_over_Rt * RT, Rd=spec.Rd_over_Rt * RT, K=spec.bell, th_n=spec.th_n, th_e=spec.th_e)

Lstar_min = L_star(e2_contour(L_cyl=0.0, **GEO))
if spec.L_star <= Lstar_min:
    raise ValueError(f"L* {spec.L_star} m is below the convergent section alone ({Lstar_min:.3f} m): "
                     "raise L*, lower CR or shorten the blend radius.")
L_CYL = brentq(lambda L: L_star(e2_contour(L_cyl=L, **GEO)) - spec.L_star, 0.0, 2.0, xtol=1e-7)
CONTOUR = e2_contour(L_cyl=L_CYL, **GEO)
WALL = wall_stations(CONTOUR, spec.n_stations)
XT = CONTOUR["xt"]

Mc, p0_loss = GD.rayleigh_p0_loss(spec.CR, DP["gam_fr"])
a_c = DP["pt"].a_c
geometry = pd.Series({
    "throat diameter  mm": 2e3 * RT, "chamber diameter  mm": 2e3 * RC,
    "exit diameter  mm": 2e3 * RT * np.sqrt(spec.eps), "contraction ratio": spec.CR,
    "cylinder length  mm": 1e3 * L_CYL, "injector face to throat  mm": 1e3 * XT,
    "throat to exit  mm": 1e3 * CONTOUR["L_nozzle"], "total length  mm": 1e3 * CONTOUR["x"][-1],
    "bell length / 15 deg cone": CONTOUR["L_nozzle"] / cone_length(RT, spec.eps, Rd_frac=spec.Rd_over_Rt),
    "blend radius R1  mm": 1e3 * GEO["R1"], "throat radii Ru / Rd  mm": f"{1e3*GEO['Ru']:.2f} / {1e3*GEO['Rd']:.2f}",
    "chamber volume  cm3": 1e6 * chamber_volume(CONTOUR), "L*  m": L_star(CONTOUR),
    "chamber Mach number": Mc, "Rayleigh stagnation pressure loss  %": 100 * p0_loss,
    "p_injector face / p_c (CEA finite-area combustor)": pinj_over_pcomb(spec.pc, round(MR_D, 5), spec.CR, OX, FUEL),
    "stay time, V rho_c / mdot  ms": 1e3 * chamber_volume(CONTOUR) * (spec.pc * 1e5 * DP["mw"] / (8314.46 * DP["Tc"])) / DP["mdot"],
    "1T acoustic mode  kHz": 1.8412 * a_c / (2 * np.pi * RC) / 1e3,
    "1L acoustic mode (face to throat)  kHz": a_c / (2 * XT) / 1e3}, name="contour")

fig, ax = plt.subplots(figsize=(11, 3.6))
x, r = 1e3 * CONTOUR["x"], 1e3 * CONTOUR["r"]
for sgn in (1, -1):
    ax.plot(x, sgn * r, color=PS.INK, lw=1.6)
ax.fill_between(x, -r, r, color=PS.BLUE, alpha=0.06)
ax.axhline(0, color=PS.MUTED, lw=0.7, ls="-.")
for xv, lab in ((0, "injector face"), (1e3 * L_CYL, "end of cylinder"), (1e3 * XT, "throat"), (x[-1], "exit")):
    ax.axvline(xv, color=PS.MUTED, lw=0.8, ls=":")
    ax.text(xv, 1.08 * r.max(), lab, fontsize=8, color=PS.INK2, ha="center")
ax.set(xlabel="x from injector face, mm", ylabel="r, mm", title=f"{spec.name}: chamber and nozzle contour")
ax.set_aspect("equal"); ax.set_ylim(-1.25 * r.max(), 1.25 * r.max())
fig.tight_layout()
geometry.to_frame()

## 8 Gas-side heat load

First cut of the thermal load with `bartz.bartz_profile()`: Bartz (1957) with frozen CEA transport
properties at chamber conditions and a prescribed gas-side wall temperature (`Twg_bartz`). The
integrated heat divided by the oxidiser flow is the enthalpy rise the coolant has to take when the
full oxidiser flow cools the chamber. The coupled solution with the real wall temperature follows
in step 9.

In [ ]:
def gas_load(s):
    b = bartz_profile(CONTOUR, s["pt"], s["pc"], Twg=spec.Twg_bartz, factor=spec.bartz_factor,
                      rc_throat=0.5 * (CONTOUR["Ru"] + CONTOUR["Rd"]))
    it = int(np.argmin(np.abs(b["x"] - XT)))
    return b, dict(pc=s["pc"], MR=s["MR"], q_throat_MW_m2=b["q"][it] / 1e6, hg_throat_W_m2K=b["hg"][it],
                   q_cylinder_MW_m2=b["q"][0] / 1e6, Q_total_kW=b["Q"] / 1e3,
                   Q_to_throat_kW=b["Q_upstream"] / 1e3, dh_per_kg_ox_kJ=b["Q"] / s["mox"] / 1e3)


B_HI, row_hi = gas_load(DP)
B_LO, row_lo = gas_load(LOW)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
for b, col, lab in ((B_HI, PS.BLUE, "100 %"), (B_LO, PS.ORANGE, tmin)):
    ax[0].plot(1e3 * b["x"], b["q"] / 1e6, color=col, label=lab)
    ax[1].plot(1e3 * b["x"], b["hg"] / 1e3, color=col, label=lab)
for a, t, u in ((ax[0], f"Heat flux, Bartz at Twg {spec.Twg_bartz:.0f} K", "MW/m²"),
                (ax[1], "Gas-side heat transfer coefficient", "kW/m²K")):
    a2 = a.twinx(); a2.plot(1e3 * CONTOUR["x"], 1e3 * CONTOUR["r"], color=PS.MUTED, lw=1); a2.grid(False)
    a2.set_ylim(0, 3.2e3 * RC); a2.set_ylabel("r, mm"); a2.spines["right"].set_visible(True)
    a.set(title=t, xlabel="x, mm", ylabel=u); a.legend()
fig.tight_layout()
pd.DataFrame({"100 %": row_hi, tmin: row_lo}).round(3)

## 9 Regenerative cooling

### 9.1 Jacket geometry

`jacket.Jacket` describes straight axial channels of constant height and constant rib thickness, so
the channel width follows the local circumference and is smallest at the throat. `jacket.channels()`
raises an error if the ribs do not fit the throat circumference.

In [ ]:
JACKET = Jacket(n=spec.n_ch, t_wall=spec.t_wall, t_rib=spec.t_rib, h=spec.h_ch,
                t_closeout=spec.t_closeout, roughness=spec.roughness, material=spec.material)
CH = channels(WALL["r"], JACKET)
xw = 1e3 * WALL["x"]
it_w = int(np.argmin(WALL["r"]))

fig, ax = plt.subplots(1, 3, figsize=(13.5, 3.4))
ax[0].plot(xw, 1e3 * CH["w"], color=PS.BLUE, label="width")
ax[0].plot(xw, 1e3 * CH["Dh"], color=PS.ORANGE, label="hydraulic diameter")
ax[0].axhline(1e3 * JACKET.h, color=PS.MUTED, ls="--", lw=1, label="height")
ax[0].set(title="Channel dimensions", xlabel="x, mm", ylabel="mm"); ax[0].legend()
ax[1].plot(xw, CH["w"] / JACKET.h, color=PS.BLUE)
ax[1].set(title="Width / height", xlabel="x, mm")
ax[2].plot(xw, DP["mox"] / CH["A_flow"], color=PS.BLUE)
ax[2].set(title="Coolant mass flux at full oxidiser flow", xlabel="x, mm", ylabel="kg/m²s")
fig.tight_layout()

pd.DataFrame({k: {"cylinder": v[0], "throat": v[it_w], "exit": v[-1]} for k, v in {
    "channel width  mm": 1e3 * CH["w"], "pitch  mm": 1e3 * CH["pitch"], "hydraulic diameter  mm": 1e3 * CH["Dh"],
    "flow area, all channels  mm2": 1e6 * CH["A_flow"],
    "land fraction t_rib / pitch": JACKET.t_rib / CH["pitch"]}.items()}).T.round(3)

### 9.2 N₂O-cooled design case

The full oxidiser flow enters the jacket at the nozzle exit with the tank enthalpy (the valves and
the venturi throttle at constant enthalpy) and leaves at the injector end. The jacket outlet
pressure is the oxidiser injector inlet pressure, scaled to part load with `n2o.jacket_pressures()`;
`regen.march_to_outlet_pressure()` finds the jacket inlet pressure that delivers it. With
`return_jacket_heat`, the N₂O card carries the jacket heat per kilogram into the chamber and the
operating point and the march are repeated until the heat is consistent. This is the procedure of
`regen.n2o_design_case()`, written out here with the `Spec` values.

Two coolant-side models bracket the result. `"regime"` switches between nucleate boiling, film
boiling and near-critical correlations and is the design case; `"liquid_only"` gives no boiling
credit but also ignores film boiling, and is optimistic for boiling N₂O.

In [ ]:
H_TANK = N2O.h_TP(spec.T_ox, spec.p_tank)        # J/kg, jacket inlet enthalpy


def n2o_case(F_frac, tp_model, label):
    # N2O-cooled operating point at a thrust fraction; returns (engine state, march result)
    extra = 0.0                                     # kJ/kg, jacket heat carried by the N2O card
    for _ in range(8):
        ox = n2o_card(spec.T_ox, spec.p_tank, round(extra, 1))
        s = op(mox_for_thrust(F_frac * spec.F, AT, MOX_D, ox=ox), AT, ox=ox)
        _, p_out = N2O.jacket_pressures(s["mox"], s["pc"], MOX_D, pc_nom=spec.pc, p_inj_nom=spec.p_inj_in)
        r = regen.march_to_outlet_pressure(WALL, s["pt"], s["pc"], JACKET, "n2o", s["mox"], p_out, H_TANK,
                                           p_guess_bar=p_out + 15, case=label, tp_model=tp_model,
                                           bartz_factor=spec.bartz_factor, eta_cstar=spec.eta_cstar,
                                           rc_throat=0.5 * (CONTOUR["Ru"] + CONTOUR["Rd"]))
        q = r["Q"] / s["mox"] / 1e3
        if not spec.return_jacket_heat or abs(q - extra) < 5.0:
            break
        extra = q
    else:
        warnings.warn(f"{label}: jacket heat not converged ({extra:.0f} -> {q:.0f} kJ/kg)")
    r.scalars["jacket_heat_returned_kJ_kg"] = extra if spec.return_jacket_heat else 0.0
    return s, r


CASES = {}
for frac in (1.0, spec.throttle_min):
    for model in dict.fromkeys((spec.tp_model, "liquid_only")):
        label = f"N2O {100 * frac:.0f} % {model}"
        CASES[label] = n2o_case(frac, model, label)
        print("done:", label)

In [ ]:
def case_row(s, r):
    sc = r.scalars
    return {"thrust  N": s["F"], "pc  bar": s["pc"], "O/F": s["MR"], "coolant flow  kg/s": sc["mdot"],
            "Isp delivered  s": s["isp"], "heat to coolant  kW": sc["Q"] / 1e3,
            "enthalpy rise  kJ/kg": (sc["h_out"] - sc["h_in"]) / 1e3,
            "jacket inlet  bar": sc["p_in_bar"], "jacket outlet  bar": sc["p_out_bar"],
            "pressure drop  bar": sc["dp_bar"], "outlet temperature  K": sc["T_out"],
            "outlet state": sc["regime_out"], "outlet quality": sc["x_out"],
            "Twg max  K": sc["Twg_max"], "at x  mm": 1e3 * sc["x_Twg_max"], "Twc max  K": sc["Twc_max"],
            "q max  MW/m2": sc["q_max"] / 1e6, "v min  m/s": sc["v_min"],
            "CHF ratio max (Hall-Mudawar)": sc["chf_ratio_max"],
            "two-phase share of the jacket": sc["frac_two_phase"], "energy balance error": sc["energy_balance_err"]}


cooling = pd.DataFrame({k: case_row(*v) for k, v in CASES.items()})
cooling.map(lambda v: round(v, 3) if isinstance(v, float) else v)

In [ ]:
def plot_cases(cases, title):
    fig, ax = plt.subplots(2, 2, figsize=(13, 7), sharex=True)
    for (label, (s, r)), col in zip(cases.items(), PS.SERIES):
        a = r.arrays
        xx = 1e3 * a["x"]
        ax[0, 0].plot(xx, a["Twg"], color=col, label=f"{label}: gas side")
        ax[0, 0].plot(xx, a["Twc"], color=col, ls="--", lw=1.3, label="coolant side")
        ax[0, 0].plot(xx, a["Tb"], color=col, ls=":", lw=1.3, label="coolant bulk")
        ax[0, 1].plot(xx, a["q"] / 1e6, color=col, label=label)
        ax[1, 0].plot(xx, a["p"] / 1e5, color=col, label=label)
        ax[1, 1].plot(xx, a["v"], color=col, label=label)
    lim = MAT.get(JACKET.material)["T_limit"]
    ax[0, 0].axhline(lim, color=PS.RED, lw=1, ls="-.", label=f"{JACKET.material} gas-side limit")
    ax[0, 0].set(title="Wall and coolant temperatures", ylabel="K")
    ax[0, 0].legend(fontsize=7, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.04))
    ax[0, 1].set(title="Heat flux", ylabel="MW/m²"); ax[0, 1].legend(fontsize=7.5)
    ax[1, 0].set(title="Coolant pressure (flow from the exit to the injector)", xlabel="x, mm", ylabel="bar")
    ax[1, 1].set(title="Coolant velocity", xlabel="x, mm", ylabel="m/s")
    for a_ in ax.ravel():
        a_.axvline(1e3 * XT, color=PS.MUTED, lw=0.8, ls=":")
    fig.suptitle(title, x=0.01, ha="left", fontsize=11, fontweight="bold")
    fig.tight_layout()


plot_cases({k: v for k, v in CASES.items() if " 100 % " in k}, f"{spec.name}: N2O-cooled, 100 % thrust")
plot_cases({k: v for k, v in CASES.items() if " 100 % " not in k}, f"{spec.name}: N2O-cooled, {tmin} thrust")

### 9.3 Checks against the handbook limits

`regen.checks()` compares each march with the limits set in `h2cea.config`: gas-side wall
temperature, coolant-side wall temperature with N₂O, minimum coolant velocity and the Hall–Mudawar
critical heat flux ratio. Saturated flow boiling in the jacket is reported as a failed check,
because the dryout limit is not evaluated by the tool. The last rows are the numerical checks of
handbook 10.8 for the design case: energy balance and the change when the station count is doubled.

In [ ]:
rows = []
for label, (s, r) in CASES.items():
    for name, (value, limit, ok) in regen.checks(r).items():
        rows.append(dict(case=label, check=name, value=value, limit=limit, ok=ok))
checks = pd.DataFrame(rows).set_index(["case", "check"])

DESIGN_CASE = f"N2O 100 % {spec.tp_model}"
s_d, r_d = CASES[DESIGN_CASE]
r2 = regen.march(wall_stations(CONTOUR, 2 * spec.n_stations), s_d["pt"], s_d["pc"], JACKET, "n2o", s_d["mox"],
                 r_d["p_in_bar"], H_TANK, tp_model=spec.tp_model, bartz_factor=spec.bartz_factor,
                 eta_cstar=spec.eta_cstar, rc_throat=0.5 * (CONTOUR["Ru"] + CONTOUR["Rd"]))
VERIFY = {"energy balance error": r_d["energy_balance_err"],
          "Twg max change with 2x stations  K": abs(r2["Twg_max"] - r_d["Twg_max"]),
          "Q change with 2x stations": abs(r2["Q"] - r_d["Q"]) / r_d["Q"],
          "pressure drop change with 2x stations": abs(r2["dp_bar"] - r_d["dp_bar"]) / r_d["dp_bar"]}
print(pd.Series(VERIFY, name=f"verification, {DESIGN_CASE}").round(4).to_string(), "\n")
checks.round(3)

### 9.4 Water-cooled test configuration (optional)

The same contour and jacket with a separate water loop, as used for the first hot fires. Set
`RUN_WATER = False` to skip it.

In [ ]:
RUN_WATER = True
if RUN_WATER:
    h_w = h_from_Tp("water", spec.water_T_in, spec.water_p_in * 1e5)
    WATER = {}
    for frac, s in ((1.0, DP), (spec.throttle_min, LOW)):
        label = f"water {100 * frac:.0f} %"
        WATER[label] = (s, regen.march(WALL, s["pt"], s["pc"], JACKET, "water", spec.water_mdot, spec.water_p_in,
                                       h_w, case=label, bartz_factor=spec.bartz_factor, eta_cstar=spec.eta_cstar,
                                       rc_throat=0.5 * (CONTOUR["Ru"] + CONTOUR["Rd"])))
    plot_cases(WATER, f"{spec.name}: water-cooled, {spec.water_mdot:g} kg/s at {spec.water_p_in:g} bar")
    display(pd.DataFrame({k: case_row(*v) for k, v in WATER.items()}).map(
        lambda v: round(v, 3) if isinstance(v, float) else v))
    display(pd.DataFrame([dict(case=k, check=n, value=v[0], limit=v[1], ok=v[2])
                          for k, (s, r) in WATER.items() for n, v in regen.checks(r).items()]
                         ).set_index(["case", "check"]).round(3))

## 10 Wall stresses

First-order stresses from `structure`: bending of the hot wall as a clamped beam over the channel,
rib tension, closeout hoop and bending for a set of pressure cases, and the thermal stress of a
fully restrained plate with the wall temperature difference of the design case. These are screening
values; a hot wall above yield in the thermal case is normal for a regeneratively cooled chamber and
sets the low-cycle fatigue life rather than a static margin.

In [ ]:
p_jacket = r_d["p_in_bar"]
LOAD_CASES = {   # name: (pressure difference across the hot wall, coolant pressure), bar
    "operating, 100 %": (p_jacket - s_d["pc"], p_jacket),
    "cold flow, chamber at ambient": (p_jacket, p_jacket),
    "tank pressure, chamber at ambient": (spec.p_tank, spec.p_tank)}
mat = MAT.get(JACKET.material)
stress = pd.DataFrame(ST.load_cases(RC, RT, JACKET, LOAD_CASES)).set_index("case")
stress["yield at 293 K  MPa"] = float(MAT.yield_strength(mat, 293.0)) / 1e6

th = ST.thermal_stress(r_d)
a = r_d.arrays
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].plot(1e3 * a["x"], th["dT_wall"], color=PS.BLUE)
ax[0].set(title=f"Wall temperature difference, {DESIGN_CASE}", xlabel="x, mm", ylabel="K")
ax[1].plot(1e3 * a["x"], th["sigma_th"] / 1e6, color=PS.BLUE, label="thermal stress, restrained plate")
ax[1].plot(1e3 * a["x"], th["yield_hot_face"] / 1e6, color=PS.RED, ls="--", label="yield at the hot-face temperature")
ax[1].set(title="Thermal stress in the hot wall", xlabel="x, mm", ylabel="MPa"); ax[1].legend()
fig.tight_layout()
print(f"Thermal stress / yield, maximum: {np.nanmax(th['ratio_to_yield']):.2f} "
      f"at x = {1e3 * a['x'][int(np.nanargmax(th['ratio_to_yield']))]:.1f} mm")
stress.round(1)

## 11 Design summary and export

The cell collects the key numbers and writes them with the inputs and the run record to
`design_summary.yaml`. It also writes the contour table for CAD (`contour_xr.csv`, millimetres, `x`
from the injector face), the throttle line and every march station of each cooling case.

In [ ]:
f4 = lambda v: round(float(v), 4)
SUMMARY = dict(
    run=dict(RUN, spec_sha=SPEC_SHA),
    spec=asdict(spec),
    design_point=dict(thrust_N=f4(DP["F"]), chamber_pressure_bar=f4(DP["pc"]), mixture_ratio=f4(DP["MR"]),
                      total_flow_kg_s=f4(DP["mdot"]), ox_flow_kg_s=f4(DP["mox"]), fuel_flow_kg_s=f4(DP["mf"]),
                      isp_delivered_s=f4(DP["isp"]), isp_ideal_s=f4(DP["isp_ideal"]),
                      cstar_ideal_m_s=f4(DP["cstar_ideal"]), cf_vac_ideal=f4(DP["cf_vac"]),
                      chamber_temperature_K=f4(DP["Tc"]), exit_pressure_bar=f4(DP["pe"])),
    throttle_floor=dict(throttle=f4(spec.throttle_min), thrust_N=f4(LOW["F"]), chamber_pressure_bar=f4(LOW["pc"]),
                        mixture_ratio=f4(LOW["MR"]), isp_delivered_s=f4(LOW["isp"]),
                        exit_pressure_bar=f4(LOW["pe"]),
                        full_flow=bool(LOW["pe"] / P_AMB > max(GD.schmucker_psep_over_pa(LOW["Me"]),
                                                               GD.summerfield_psep_over_pa()))),
    geometry=dict(throat_diameter_mm=f4(2e3 * RT), chamber_diameter_mm=f4(2e3 * RC),
                  exit_diameter_mm=f4(2e3 * RT * np.sqrt(spec.eps)), cylinder_length_mm=f4(1e3 * L_CYL),
                  face_to_throat_mm=f4(1e3 * XT), throat_to_exit_mm=f4(1e3 * CONTOUR["L_nozzle"]),
                  total_length_mm=f4(1e3 * CONTOUR["x"][-1]), L_star_m=f4(L_star(CONTOUR)),
                  blend_radius_mm=f4(1e3 * GEO["R1"]), throat_radius_up_mm=f4(1e3 * GEO["Ru"]),
                  throat_radius_down_mm=f4(1e3 * GEO["Rd"])),
    gas_side_bartz=dict(wall_temperature_K=spec.Twg_bartz, **{k: f4(v) for k, v in row_hi.items()}),
    cooling={label: dict(heat_kW=f4(r["Q"] / 1e3), jacket_inlet_bar=f4(r["p_in_bar"]),
                         pressure_drop_bar=f4(r["dp_bar"]), outlet_temperature_K=f4(r["T_out"]),
                         outlet_state=str(r["regime_out"]), Twg_max_K=f4(r["Twg_max"]), Twc_max_K=f4(r["Twc_max"]),
                         jacket_heat_returned_kJ_kg=f4(r["jacket_heat_returned_kJ_kg"]),
                         isp_delivered_s=f4(s["isp"]),
                         checks_passed=bool(all(v[2] for v in regen.checks(r).values())))
             for label, (s, r) in CASES.items()},
    verification={k: f4(v) for k, v in VERIFY.items()},
    stress_MPa={c: {k: f4(v) for k, v in row.items() if k.endswith("MPa")} for c, row in stress.iterrows()},
)

(OUT / "design_summary.yaml").write_text(yaml.safe_dump(SUMMARY, sort_keys=False, allow_unicode=True), encoding="utf-8")
pd.DataFrame({"x_mm": 1e3 * CONTOUR["x"], "r_mm": 1e3 * CONTOUR["r"]}).to_csv(OUT / "contour_xr.csv", index=False, float_format="%.5f")
line.to_csv(OUT / "throttle_line.csv")
for label, (s, r) in {**CASES, **(WATER if RUN_WATER else {})}.items():
    r.dataframe().to_csv(OUT / f"march_{label.replace(' %', 'pct').replace(' ', '_')}.csv", index=False)

print("Written to", OUT)
for p in sorted(OUT.iterdir()):
    print("  ", p.name)

headline = pd.Series({
    "design": f"{spec.name} (spec {SPEC_SHA}, h2cea {RUN['h2cea_commit']})",
    "thrust / chamber pressure": f"{DP['F']:.0f} N at {DP['pc']:.1f} bar, O/F {DP['MR']:.2f}",
    "flows": f"{DP['mdot']:.3f} kg/s (oxidiser {DP['mox']:.3f}, fuel {DP['mf']:.3f})",
    "specific impulse, sea level": f"{DP['isp']:.1f} s delivered, {DP['isp_ideal']:.1f} s ideal",
    "throat / chamber / exit diameter": f"{2e3*RT:.2f} / {2e3*RC:.2f} / {2e3*RT*np.sqrt(spec.eps):.2f} mm",
    "length": f"{1e3*XT:.1f} mm to the throat, {1e3*CONTOUR['x'][-1]:.1f} mm total, L* {L_star(CONTOUR):.2f} m",
    f"throttle floor ({tmin})": f"pc {LOW['pc']:.1f} bar, O/F {LOW['MR']:.2f}, pe/pa {LOW['pe']/P_AMB:.2f}",
    "Bartz throat heat flux": f"{row_hi['q_throat_MW_m2']:.1f} MW/m² at Twg {spec.Twg_bartz:.0f} K",
    f"design case ({DESIGN_CASE})": f"Twg max {r_d['Twg_max']:.0f} K, Twc max {r_d['Twc_max']:.0f} K, "
                                   f"jacket inlet {r_d['p_in_bar']:.1f} bar, {r_d['Q']/1e3:.0f} kW",
    "limits met by the design case": "yes" if all(v[2] for v in regen.checks(r_d).values()) else "no, see step 9.3",
}, name="summary")
headline.to_frame()

## Model limits

State these in any design document that uses numbers from this notebook (see `h2cea/README.md`).

- **Performance.** CEA shifting equilibrium with assumed efficiencies; replace `eta_cstar` and
  `eta_cf_vac` with hot-fire values. The sizing with a fixed fuel flow searches the mixture ratio
  between 2.5 and 6 (`operating_line.size_throat`).
- **Contour.** The Rao angles are inputs, and the bell is a parabola approximation of the optimum
  contour. The contraction ratio, characteristic length and blend radii are design choices; this
  notebook does not assess combustion stability or injector performance.
- **Gas side.** Bartz with frozen CEA transport and the throat mass flux of the delivered engine.
  The adiabatic wall temperature uses the ideal chamber temperature, which is conservative. Film
  cooling and soot are not modelled. Calibrate `bartz_factor` with calorimetry.
- **Boiling N₂O.** The regime model combines correlations that are not validated for N₂O in
  mini-channels at these conditions. Read its result as "film boiling is likely", not as an accurate
  wall temperature. The liquid-only model underpredicts the wall temperature. Dryout in saturated
  flow boiling is not evaluated, and Hall–Mudawar is used only up to a quality of 0.05.
- **N₂O transport properties** come from CO₂ corresponding states, about 10 to 15 % away from the
  critical point and worse near it.
- **Jacket.** Constant channel height and rib thickness, one pass from the exit to the injector.
  Manifold, volute and inlet losses are not in the pressure drop. The heat returned to the chamber
  assumes that all of it reaches the injector.
- **Stresses.** Closed-form screening values only.